# MIMIC multi-task trainer — medications, missing columns, vitals

Runs unchanged on **Kaggle**, **Google Colab**, or a local machine, and uses **every GPU it
can see**. One shared mask-aware encoder feeds three heads:

| head | question | metric |
|---|---|---|
| `drug` | which medication classes are active in the next hour, from vitals alone | AUROC / average precision |
| `impute` | the current value of the vitals that were **not** supplied | MAE vs. median-fill |
| `forecast` | the next hour of every vital | MAE vs. repeat-last-value |

Run the cells in order. Cell 1 is the only one that asks for anything.

> Research prototype on open demo cohorts. **Not for clinical use.**

In [ ]:
# 1. Pick the epoch count and inspect the runtime
import os, shutil, subprocess, sys, platform
from pathlib import Path

def run_tool(name, *args):
    """Run an optional command-line tool. Absent is a reported fact, not a crash."""
    if shutil.which(name) is None:
        print(f'{name}: not installed on this runtime')
        return False
    return subprocess.run([name, *args], check=False).returncode == 0

def ask_positive_int(name, prompt, default):
    """Env var wins, then an interactive prompt, then the default in batch mode."""
    preset = os.environ.get(name)
    if not preset:
        batch = (os.environ.get('NONINTERACTIVE') == '1'
                 or os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '').lower() in {'batch', 'scheduled'})
        if batch:
            print(f'{name}: non-interactive runtime, using default {default}')
            return default
        try:
            preset = input(f'{prompt} [{default}]: ').strip()
        except (EOFError, OSError):
            preset = ''
    try:
        value = int(preset)
    except (TypeError, ValueError):
        return default
    return value if value > 0 else default

TRAIN_EPOCHS = ask_positive_int('TRAIN_EPOCHS', 'Training epochs', 60)

print(f'epochs      : {TRAIN_EPOCHS}')
print(f'python      : {platform.python_version()} on {platform.system()}')
print(f'cpu cores   : {os.cpu_count()}')
if not run_tool('nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'):
    print('no NVIDIA driver visible; cell 3 confirms what torch can actually use')

In [ ]:
# 2. Locate the project, or clone it when PROJECT_REPO_URL is set
PROJECT_NAME = 'MIMC-LLM-Time-Series-Forecaster'
candidates = [Path.cwd(), Path.cwd() / PROJECT_NAME,
              Path('/kaggle/working') / PROJECT_NAME, Path('/content') / PROJECT_NAME,
              Path.home() / PROJECT_NAME]
PROJECT_DIR = next((p.resolve() for p in candidates
                    if (p / 'pyproject.toml').exists() and (p / 'src/mimic_forecaster').is_dir()), None)
if PROJECT_DIR is None:
    url = os.environ.get('PROJECT_REPO_URL')
    if not url:
        raise SystemExit('Project not found. Upload it, or set PROJECT_REPO_URL to clone it.')
    target = (Path('/kaggle/working') if Path('/kaggle/working').exists()
              else Path('/content') if Path('/content').exists() else Path.cwd()) / PROJECT_NAME
    subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)
    PROJECT_DIR = target.resolve()
os.chdir(PROJECT_DIR)
print('project:', PROJECT_DIR)

In [ ]:
# 3. Install the package and report every visible GPU
import importlib.util
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)

import numpy as np, torch, yaml

GPUS = torch.cuda.device_count()
if GPUS:
    for index in range(GPUS):
        properties = torch.cuda.get_device_properties(index)
        print(f'  cuda:{index}  {properties.name}  {properties.total_memory / 1e9:.1f} GB')
    # One batch is split across devices by DataParallel, so scale it with the device count
    # to keep the per-GPU batch (and therefore the step count) the same.
    # PER_GPU_BATCH fills more memory per card. The model is small, so 128 already saturates
    # compute; raising it mostly cuts the number of optimizer steps on a 276-stay cohort.
    PER_GPU_BATCH = int(os.environ.get('PER_GPU_BATCH', 128))
    BATCH_SIZE, WORKERS = PER_GPU_BATCH * GPUS, min(4 * GPUS, os.cpu_count() or 2)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True   # Ampere+ gets free throughput here
    torch.backends.cudnn.allow_tf32 = True
else:
    print('  no CUDA device: training on CPU (correct, just slower)')
    BATCH_SIZE, WORKERS = 64, min(4, os.cpu_count() or 1)
print(f'torch {torch.__version__}  |  gpus {GPUS}  |  batch {BATCH_SIZE}  |  workers {WORKERS}')

In [ ]:
# 4. Fetch the open MIMIC-IV and MIMIC-III demos only when they are absent
DEMOS = {
    'mimic-iv-demo-2.2': ('https://physionet.org/files/mimic-iv-demo/2.2/',
                          'physionet.org/files/mimic-iv-demo/2.2/icu/chartevents.csv.gz'),
    'mimic-iii-demo-1.4': ('https://physionet.org/files/mimiciii-demo/1.4/',
                           'physionet.org/files/mimiciii-demo/1.4/CHARTEVENTS.csv'),
}
for name, (url, probe) in DEMOS.items():
    destination = PROJECT_DIR / 'data/raw' / name
    if (destination / probe).exists():
        print(f'{name}: already present')
        continue
    destination.mkdir(parents=True, exist_ok=True)
    print(f'{name}: downloading (open-access demo, no credentials needed)')
    if shutil.which('wget') is None:
        raise SystemExit('wget is not installed. Install it, or upload data/raw/ with the project.')
    subprocess.run(['wget', '-r', '-N', '-c', '-np', '-q', '--show-progress', url],
                   cwd=destination, check=False)
    print(f'{name}: {"ok" if (destination / probe).exists() else "FAILED - check network"}')

In [ ]:
# 5. Write a runtime config, leaving the tracked example files untouched
AVAILABLE = [path for path in (PROJECT_DIR / 'data/raw').glob('mimic-i*-demo*') if path.is_dir()]
base_name = 'mimic_pooled.yaml' if len(AVAILABLE) > 1 else 'mimic_iv_demo.yaml'
with (PROJECT_DIR / 'configs' / base_name).open() as handle:
    config = yaml.safe_load(handle)

config['project']['device'] = 'auto'          # every visible CUDA GPU, else MPS, else CPU
config['project']['artifact_dir'] = 'artifacts/runtime'
config['training']['epochs'] = TRAIN_EPOCHS
config['training']['batch_size'] = BATCH_SIZE
config['training']['num_workers'] = WORKERS
config['training']['amp'] = bool(GPUS)        # mixed precision only where it is supported
config['training']['patience'] = max(5, TRAIN_EPOCHS // 6)

# Drop any source whose files are not on this machine, so a partial download still trains.
if 'sources' in config['data']:
    kept = []
    for source in config['data']['sources']:
        root = PROJECT_DIR / source['root']
        if (root / source['icustays']).exists() and (root / source['chartevents']).exists():
            if source.get('prescriptions') and not (root / source['prescriptions']).exists():
                print(f"{source['name']}: no prescriptions file, medication labels will be masked out")
                source.pop('prescriptions')
            kept.append(source)
        else:
            print(f"{source['name']}: files absent, dropping from this run")
    if not kept:
        raise SystemExit('No configured source has its files on disk.')
    config['data']['sources'] = kept

RUNTIME_CONFIG = PROJECT_DIR / 'configs/runtime.yaml'
with RUNTIME_CONFIG.open('w') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)
print(yaml.safe_dump({k: config[k] for k in ('project', 'task', 'training')}, sort_keys=False))

In [ ]:
# 6. Inspect the schema, then build leakage-safe windows with medication labels
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_DIR / 'src')
env['PYTHONDONTWRITEBYTECODE'] = '1'

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('>', ' '.join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)

run_module('mimic_forecaster.cli.inspect', '--config', RUNTIME_CONFIG)
run_module('mimic_forecaster.cli.prepare', '--config', RUNTIME_CONFIG)

In [ ]:
# 7. Train the shared encoder with all three heads, on every available GPU
run_module('mimic_forecaster.cli.train', '--config', RUNTIME_CONFIG)

In [ ]:
# 8. Evaluate on the untouched subject-level test split
run_module('mimic_forecaster.cli.evaluate', '--config', RUNTIME_CONFIG)

## 9. The missing-column path

Supply only the vitals a site actually charts. Everything else is marked absent exactly as
it would be at a source that never records it, and comes back under `imputed_columns`,
alongside the predicted medication classes.

In [ ]:
# 9. Predict from a partial column set
run_module('mimic_forecaster.cli.predict', '--config', RUNTIME_CONFIG,
           '--limit-stays', 1, '--features', 'heart_rate,spo2,respiratory_rate')

## 9b. Predict for your own patient

Give a CSV with a `time` column and any subset of `heart_rate, respiratory_rate, spo2,
systolic_bp, diastolic_bp, mean_bp, temperature_f`, covering at least 24 hours. Columns you
leave out come back under `imputed_columns`; `medications.predicted` lists the drug classes
the model expects in the next hour. See `examples/patient.csv` for the format.

- **Colab**: upload the file in the file browser, then set `PATIENT_CSV` to its path.
- **Kaggle**: *Add Data ▸ Upload* it, then set `PATIENT_CSV` to the `/kaggle/input/...` path.


In [ ]:
# 9b. Predict medications for your own patient
import json
PATIENT_CSV = os.environ.get('PATIENT_CSV', str(PROJECT_DIR / 'examples/patient.csv'))
command = [sys.executable, '-m', 'mimic_forecaster.cli.predict', '--config', str(RUNTIME_CONFIG),
           '--patient-csv', PATIENT_CSV]
result = json.loads(subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True,
                                   capture_output=True, text=True).stdout)
patient = result['predictions'][0]
probability = patient['medications']['probability']
print('supplied :', result['supplied_columns'])
print('imputed  :', patient['imputed_columns'])
print('next hour:', patient['forecast'][0]['values'])
print('\nmedication classes, most likely first:')
for name in sorted(probability, key=probability.get, reverse=True):
    print(f'  {probability[name]:.2f}  {name}')


In [ ]:
# 10. Collect the artifacts before the session ends
import json, shutil
artifacts = PROJECT_DIR / config['project']['artifact_dir']
print(json.dumps(json.loads((artifacts / 'test_metrics.json').read_text()), indent=2)[:4000])
archive = shutil.make_archive(str(PROJECT_DIR / 'mimic_multitask_artifacts'), 'zip', artifacts)
print('\nartifacts ->', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print('Kaggle: this file appears under the notebook Output tab. Colab: download it from the file browser.')

## What is in `artifacts/runtime/`

| file | contents |
|---|---|
| `prepared/windows.npz` | windows, masks, vital targets, medication labels |
| `prepared/preprocessing.json` | training-fitted robust scaler |
| `prepared/metadata.json` | feature and drug registries, digests, label prevalence |
| `checkpoints/multitask.pt` | best-validation weights plus the contract they were trained under |
| `training_metrics.json` | per-epoch loss, split out per head |
| `test_metrics.json` | the three evaluation sections |

A checkpoint refuses to load against a different config or registry digest, so a run can
never be evaluated against data it was not trained on.